# EMC Event Reservation — Forecasting Models

This notebook covers:
1. Loading the cleaned monthly time series
2. Stationarity test (ADF Test) — required for SARIMA
3. Train / Test split (18 months train, 4 months test)
4. Building 3 forecasting models:
   - Model 1: Moving Average
   - Model 2: Linear Regression
   - Model 3: SARIMA
5. Computing MAE and RMSE for each model
6. Final comparison table and winner

**Input file:** `EMC_Monthly_Reservations.csv`

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries if not already installed
# Run this cell once, then you can skip it on future runs
import subprocess
subprocess.run(['pip', 'install', 'statsmodels', 'scikit-learn', '--quiet'], check=False)
print('Libraries ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Stats & ML
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

print('All libraries imported successfully.')

## Step 2: Load the Monthly Reservation Data

In [ ]:
# Update path if your file is in a different folder
FILE_PATH = 'EMC_Monthly_Reservations.csv'

df = pd.read_csv(FILE_PATH)
df['month'] = pd.to_datetime(df['month'])
df = df.set_index('month')
df.index.freq = 'ME'  # Monthly end frequency

print(f'Loaded {len(df)} months of data')
print(f'Date range: {df.index[0].strftime("%b %Y")}  →  {df.index[-1].strftime("%b %Y")}')
print()
print(df)

## Step 3: Visualize the Full Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df['reservation_count'], marker='o', color='steelblue', linewidth=2)
ax.fill_between(df.index, df['reservation_count'], alpha=0.15, color='steelblue')
ax.set_title('Monthly Event Reservations — EMC Logbook', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_full_series.png', dpi=150)
plt.show()

## Step 4: Stationarity Test (ADF Test)

**Why:** SARIMA requires the data to be stationary (stable mean and variance over time).

**Interpretation:**
- p-value < 0.05 → Data IS stationary → SARIMA can proceed as-is
- p-value ≥ 0.05 → Data is NOT stationary → We apply differencing (d=1) in SARIMA parameters

In [ ]:
def run_adf_test(series, label='Series'):
    result = adfuller(series.dropna())
    print(f'=== ADF Test: {label} ===')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical Values:')
    for key, val in result[4].items():
        print(f'    {key}: {val:.4f}')
    if result[1] < 0.05:
        print('  ✅ STATIONARY (p < 0.05) — SARIMA can proceed as-is')
        return True
    else:
        print('  ⚠️  NOT STATIONARY (p ≥ 0.05) — Use d=1 in SARIMA to difference the data')
        return False

is_stationary = run_adf_test(df['reservation_count'], 'Monthly Reservations')

In [ ]:
# If not stationary, also test the differenced series (for reference)
if not is_stationary:
    print()
    print('Testing differenced series (lag-1):')
    run_adf_test(df['reservation_count'].diff(), 'Differenced Series')
    print()
    print('→ We will set d=1 in our SARIMA model to handle non-stationarity automatically.')
else:
    print()
    print('→ We will set d=0 in our SARIMA model (no differencing needed).')

## Step 5: Train / Test Split

We split the 22 months of data as follows:
- **Training set:** First 18 months (Jan 2024 – Jun 2025) — models learn from this
- **Test set:** Last 4 months (Jul 2025 – Oct 2025) — we evaluate predictions against this

This is an 82% / 18% split, which is standard for short time series.

In [ ]:
TEST_SIZE = 4  # number of months held out for testing

train = df.iloc[:-TEST_SIZE]
test  = df.iloc[-TEST_SIZE:]

print(f'Training set : {len(train)} months  ({train.index[0].strftime("%b %Y")} → {train.index[-1].strftime("%b %Y")})')
print(f'Test set     : {len(test)} months  ({test.index[0].strftime("%b %Y")} → {test.index[-1].strftime("%b %Y")})')
print()
print('Training data:')
print(train)
print()
print('Test data (what we will predict):')
print(test)

In [ ]:
# Visualize the split
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train['reservation_count'], marker='o', color='steelblue',
        linewidth=2, label='Training Set')
ax.plot(test.index, test['reservation_count'], marker='o', color='tomato',
        linewidth=2, label='Test Set')
ax.axvline(x=train.index[-1], color='gray', linestyle='--', linewidth=1.5, label='Split Point')
ax.set_title('Train / Test Split', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_train_test_split.png', dpi=150)
plt.show()

## Step 6: Helper — MAE & RMSE Function

In [ ]:
def compute_errors(actual, predicted, model_name):
    """Compute and print MAE and RMSE for a model."""
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    print(f'  MAE  = {mae:.4f}   (on average, off by {mae:.1f} reservations per month)')
    print(f'  RMSE = {rmse:.4f}   (penalizes large errors more)')
    return {'Model': model_name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4)}

# Store results from all models here
results = []

---
## Model 1: Moving Average

**How it works:** Predicts the next month as the average of the last N months (rolling window).

**Best for:** Smoothing out noise. Good simple baseline.

In [ ]:
# Window size — using last 3 months average
# You can try 2, 3, 4, 6 and see which gives lower MAE/RMSE
MA_WINDOW = 3

# Generate predictions for each test month
# For each test step, we average the last MA_WINDOW months from training + already-predicted months
history = list(train['reservation_count'].values)
ma_predictions = []

for i in range(TEST_SIZE):
    window_avg = np.mean(history[-MA_WINDOW:])
    ma_predictions.append(window_avg)
    # Use actual test value as next step's history (walk-forward)
    history.append(test['reservation_count'].iloc[i])

ma_pred_series = pd.Series(ma_predictions, index=test.index)

print(f'=== Model 1: Moving Average (window={MA_WINDOW}) ===')
print(f'  Month       Actual   Predicted')
for i, idx in enumerate(test.index):
    print(f'  {idx.strftime("%b %Y")}    {test["reservation_count"].iloc[i]:>6}   {ma_predictions[i]:>9.2f}')
print()
ma_result = compute_errors(test['reservation_count'], ma_predictions, f'Moving Average (w={MA_WINDOW})')
results.append(ma_result)

In [ ]:
# Plot Moving Average predictions vs actuals
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train['reservation_count'], marker='o', color='steelblue', label='Training')
ax.plot(test.index, test['reservation_count'], marker='o', color='tomato', label='Actual (Test)')
ax.plot(test.index, ma_pred_series, marker='s', linestyle='--', color='darkorange', label=f'MA Predicted (w={MA_WINDOW})')
ax.axvline(x=train.index[-1], color='gray', linestyle='--', linewidth=1)
ax.set_title(f'Model 1: Moving Average (window={MA_WINDOW})', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_model1_moving_average.png', dpi=150)
plt.show()

---
## Model 2: Linear Regression

**How it works:** Treats time as a numeric variable (1, 2, 3, ...) and fits a straight line through the data to find the trend. Then extrapolates that line into the future.

**Best for:** Data with a clear upward or downward trend.

In [ ]:
from sklearn.linear_model import LinearRegression

# Create numeric time index (1 = Jan 2024, 2 = Feb 2024, etc.)
train_idx = np.arange(1, len(train) + 1).reshape(-1, 1)
test_idx  = np.arange(len(train) + 1, len(train) + TEST_SIZE + 1).reshape(-1, 1)

lr_model = LinearRegression()
lr_model.fit(train_idx, train['reservation_count'])

lr_predictions = lr_model.predict(test_idx)
lr_pred_series = pd.Series(lr_predictions, index=test.index)

print(f'=== Model 2: Linear Regression ===')
print(f'  Slope     : {lr_model.coef_[0]:.4f}  (reservations change per month)')
print(f'  Intercept : {lr_model.intercept_:.4f}')
print()
print(f'  Month       Actual   Predicted')
for i, idx in enumerate(test.index):
    print(f'  {idx.strftime("%b %Y")}    {test["reservation_count"].iloc[i]:>6}   {lr_predictions[i]:>9.2f}')
print()
lr_result = compute_errors(test['reservation_count'], lr_predictions, 'Linear Regression')
results.append(lr_result)

In [ ]:
# Plot Linear Regression predictions vs actuals
all_idx = np.arange(1, len(df) + 1).reshape(-1, 1)
all_preds = lr_model.predict(all_idx)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train['reservation_count'], marker='o', color='steelblue', label='Training')
ax.plot(test.index, test['reservation_count'], marker='o', color='tomato', label='Actual (Test)')
ax.plot(df.index, all_preds, linestyle='--', color='gray', alpha=0.5, label='Regression Line (full)')
ax.plot(test.index, lr_pred_series, marker='s', linestyle='--', color='purple', label='LR Predicted')
ax.axvline(x=train.index[-1], color='gray', linestyle='--', linewidth=1)
ax.set_title('Model 2: Linear Regression', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_model2_linear_regression.png', dpi=150)
plt.show()

---
## Model 3: SARIMA

**How it works:** SARIMA stands for Seasonal AutoRegressive Integrated Moving Average. It captures trend, seasonality, and autocorrelation (how past values influence future ones) all at once.

**Parameters used:**
- `(p, d, q)` — Non-seasonal: autoregression, differencing, moving average
- `(P, D, Q, s)` — Seasonal: same but for seasonal patterns, s=12 for monthly

**Note:** d=1 is used if data was non-stationary in the ADF test above. Otherwise d=0.

In [ ]:
# Set d based on ADF test result above
d_value = 0 if is_stationary else 1

# SARIMA parameters — (p,d,q) x (P,D,Q,s)
# These are conservative settings suitable for 18-month training data
# p=1: use last 1 month's value
# d=auto from ADF test
# q=1: use last 1 month's error
# P=1, D=1, Q=1, s=12: annual seasonality
ORDER         = (1, d_value, 1)   # (p, d, q)
SEASONAL_ORDER = (1, 1, 1, 12)    # (P, D, Q, s)

print(f'SARIMA order          : {ORDER}')
print(f'SARIMA seasonal order : {SEASONAL_ORDER}')
print(f'(d={d_value} based on ADF test — {"stationary" if is_stationary else "non-stationary"})')
print()
print('Fitting SARIMA model...')

sarima_model = SARIMAX(
    train['reservation_count'],
    order=ORDER,
    seasonal_order=SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print('SARIMA model fitted successfully.')
print()
print(sarima_fit.summary())

In [ ]:
# Forecast the test period
sarima_forecast = sarima_fit.forecast(steps=TEST_SIZE)
sarima_pred_series = pd.Series(sarima_forecast.values, index=test.index)

print(f'=== Model 3: SARIMA {ORDER} x {SEASONAL_ORDER} ===')
print(f'  Month       Actual   Predicted')
for i, idx in enumerate(test.index):
    print(f'  {idx.strftime("%b %Y")}    {test["reservation_count"].iloc[i]:>6}   {sarima_forecast.values[i]:>9.2f}')
print()
sarima_result = compute_errors(test['reservation_count'], sarima_forecast.values, f'SARIMA{ORDER}x{SEASONAL_ORDER}')
results.append(sarima_result)

In [ ]:
# Plot SARIMA predictions vs actuals
# Get confidence intervals for the forecast
forecast_obj = sarima_fit.get_forecast(steps=TEST_SIZE)
conf_int = forecast_obj.conf_int()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train.index, train['reservation_count'], marker='o', color='steelblue', label='Training')
ax.plot(test.index, test['reservation_count'], marker='o', color='tomato', label='Actual (Test)')
ax.plot(test.index, sarima_pred_series, marker='s', linestyle='--', color='green', label='SARIMA Predicted')
ax.fill_between(test.index,
                conf_int.iloc[:, 0],
                conf_int.iloc[:, 1],
                color='green', alpha=0.15, label='95% Confidence Interval')
ax.axvline(x=train.index[-1], color='gray', linestyle='--', linewidth=1)
ax.set_title(f'Model 3: SARIMA {ORDER} x {SEASONAL_ORDER}', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_model3_sarima.png', dpi=150)
plt.show()

---
## Step 7: Final Comparison — MAE & RMSE

In [ ]:
results_df = pd.DataFrame(results).sort_values('MAE')
results_df['Rank (by MAE)'] = range(1, len(results_df) + 1)

print('=' * 60)
print('         FINAL MODEL COMPARISON — MAE & RMSE')
print('=' * 60)
print(results_df.to_string(index=False))
print('=' * 60)
print()
best_model = results_df.iloc[0]
print(f'🏆 BEST MODEL: {best_model["Model"]}')
print(f'   MAE  = {best_model["MAE"]}  (avg error per month)')
print(f'   RMSE = {best_model["RMSE"]}  (penalizes large errors)')
print()
print('Lower MAE and RMSE = better accuracy.')
print('Use the best model for your final reservation forecast.')

In [ ]:
# Bar chart comparison of MAE and RMSE
models   = results_df['Model'].tolist()
mae_vals  = results_df['MAE'].tolist()
rmse_vals = results_df['RMSE'].tolist()

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, mae_vals,  width, label='MAE',  color='steelblue')
bars2 = ax.bar(x + width/2, rmse_vals, width, label='RMSE', color='tomato')

ax.set_title('Model Comparison — MAE & RMSE\n(Lower is Better)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('Error Value')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

# Label bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150)
plt.show()

In [ ]:
# All models on one chart vs actual test values
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(train.index, train['reservation_count'], marker='o',
        color='steelblue', linewidth=2, label='Training Data')
ax.plot(test.index, test['reservation_count'], marker='o',
        color='black', linewidth=2, markersize=8, label='Actual (Test)')
ax.plot(test.index, ma_pred_series,      marker='s', linestyle='--', color='darkorange', label=f'Moving Average (w={MA_WINDOW})')
ax.plot(test.index, lr_pred_series,      marker='^', linestyle='--', color='purple',     label='Linear Regression')
ax.plot(test.index, sarima_pred_series,  marker='D', linestyle='--', color='green',      label='SARIMA')

ax.axvline(x=train.index[-1], color='gray', linestyle='--', linewidth=1.5, label='Train/Test Split')
ax.set_title('All Models vs Actual — Test Period', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Reservation Count')
ax.legend(loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plot_all_models_comparison.png', dpi=150)
plt.show()

## Step 8: Save Results to CSV

In [ ]:
# Save the comparison table
results_df.to_csv('model_comparison_results.csv', index=False)

# Save predictions vs actuals for all models
predictions_df = pd.DataFrame({
    'month':              test.index.strftime('%Y-%m'),
    'actual':             test['reservation_count'].values,
    'moving_average':     np.round(ma_predictions, 2),
    'linear_regression':  np.round(lr_predictions, 2),
    'sarima':             np.round(sarima_forecast.values, 2),
})
predictions_df.to_csv('model_predictions.csv', index=False)

print('Saved:')
print('  model_comparison_results.csv  ← MAE & RMSE table for all models')
print('  model_predictions.csv         ← Actual vs Predicted for all models')
print()
print('Plots saved:')
for f in ['plot_full_series', 'plot_train_test_split',
          'plot_model1_moving_average', 'plot_model2_linear_regression',
          'plot_model3_sarima', 'plot_model_comparison', 'plot_all_models_comparison']:
    print(f'  {f}.png')